<a href="https://colab.research.google.com/github/32220/Industrial-Predictive-Maintenance-Dashboard/blob/main/Predictive_Maintenance_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from enum import unique
import pandas as sh
x = sh.read_csv("predictive_maintenance_v3.csv")
x.head()
x["machine_id"].unique()
x["machine_id"].value_counts()
x.groupby("machine_id")["machine_type"].unique()
x["temperature_motor"].max()
x["temperature_motor"].min()
x.agg(['min','max'])
x.head()
x["machine_id"].unique()
valid = x[
    (x["failure_within_24h"] == 0) &
    (
        (x["failure_type"] != "None") &
        (x["estimated_repair_cost"] == 0)
    )
]
if len(valid) == (x["failure_within_24h"] == 0).sum():
    print("All records satisfy the rule.")
else:
    print(f"{len(valid)-(x["failure_within_24h"] == 0).sum()} records violate the rule.")
invalid = x[
    (x["failure_within_24h"] == 1) &
    (
        (x["estimated_repair_cost"] != 0)
    )
]
if len(invalid) == (x["failure_within_24h"] == 1).sum():
    print("All records satisfy the rule.")
else:
    print(f"{len(invalid)} records violate the rule.")

import pandas as pd
from sklearn.ensemble import RandomForestRegressor

features = [
    "current_phase_avg",
    "vibration_rms",
    "pressure_level",
    "rpm"
]

train = x[x["temperature_motor"].notna()]
test = x[x["temperature_motor"].isna()]
X_train = train[features]
y_train = train["temperature_motor"]
X_test = test[features]
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
x.loc[x["temperature_motor"].isna(), "temperature_motor"] = model.predict(X_test)

features = [
    "current_phase_avg",
    "temperature_motor",
    "pressure_level",
    "rpm"
]

train = x[x["vibration_rms"].notna()]
test = x[x["vibration_rms"].isna()]
X_train = train[features]
y_train = train["vibration_rms"]
X_test = test[features]
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
x.loc[x["vibration_rms"].isna(), "vibration_rms"] = model.predict(X_test)

features = [
    "vibration_rms",
    "temperature_motor",
    "pressure_level",
    "rpm"
]

train = x[x["current_phase_avg"].notna()]
test = x[x["current_phase_avg"].isna()]
X_train = train[features]
y_train = train["current_phase_avg"]
X_test = test[features]
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
x.loc[x["current_phase_avg"].isna(), "current_phase_avg"] = model.predict(X_test)

features = [
    "vibration_rms",
    "temperature_motor",
    "current_phase_avg",
    "rpm"
]
train = x[x["pressure_level"].notna()]
test = x[x["pressure_level"].isna()]
X_train = train[features]
y_train = train["pressure_level"]
X_test = test[features]
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
x.loc[x["pressure_level"].isna(), "pressure_level"] = model.predict(X_test)
features = [
    "vibration_rms",
    "temperature_motor",
    "current_phase_avg",
    "pressure_level"
]

train = x[x["rpm"].notna()]
test = x[x["rpm"].isna()]
X_train = train[features]
y_train = train["rpm"]
X_test = test[features]
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
x.loc[x["rpm"].isna(), "rpm"] = model.predict(X_test)

print(x.isnull().sum())

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

features = [
    "temperature_motor",
    "current_phase_avg",
    "pressure_level",
    "rpm"
]

data = x[x["vibration_rms"].notna()]
X = data[features]
y = data["vibration_rms"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

x["temp_difference"] = (
    x["temperature_motor"] - x["ambient_temp"] + 10
)
x["temp_status"] = pd.cut(
    x["temp_difference"],
    bins=[0,60,80,105,125,float("inf")],
    labels=["Normal","Caution","Warning","Alert","Critical"]
)
x["maintenance_risk"] = (
    x["hours_since_maintenance"] *
    x["vibration_rms"]
)

x["maintenance_status"] = pd.cut(
    x["hours_since_maintenance"],
    bins=[0,100,300,600,float("inf")],
    labels=["Recent","Normal","Overdue","Critical"]
)
from sklearn.preprocessing import MinMaxScaler

features = [
    "current_phase_avg",
    "temperature_motor",
    "temp_difference",
    "vibration_rms"
]

scaler = MinMaxScaler()

x[[f + "_scaled" for f in features]] = scaler.fit_transform(x[features])
x["machine_health_index"] = 100 * (
    1
    - (
        0.35 * x["vibration_rms_scaled"] +
        0.30 * x["temperature_motor_scaled"] +
        0.20 * x["current_phase_avg_scaled"] +
        0.15 * x["temp_difference_scaled"]
    )
)
x["machine_health_status"] = pd.cut(
    x["machine_health_index"],
    bins=[0, 40, 60, 80, 100],
    labels=["Critical", "Poor", "Good", "Excellent"],
    include_lowest=True
)

x.head()
x.to_excel("Cleaned_Predictive_Maintenance.xlsx", index=False)

All records satisfy the rule.
All records satisfy the rule.
timestamp                  0
machine_id                 0
machine_type               0
vibration_rms              0
temperature_motor          0
current_phase_avg          0
pressure_level             0
rpm                        0
operating_mode             0
hours_since_maintenance    0
ambient_temp               0
rul_hours                  0
failure_within_24h         0
failure_type               0
estimated_repair_cost      0
dtype: int64
MAE : 0.2202088548554793
RMSE: 0.43266104158666785
R²  : 0.8411931273848992


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
